In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
import pandas as pd
from faker import Faker
import random
import uuid
import json
import os

# Initialize Faker and set seeds for reproducibility
fake = Faker()
Faker.seed(42)
random.seed(42)

TOTAL_RECORDS = 5000

print(f"Generating {TOTAL_RECORDS} base 'Truth' records...")

# 1. Generate the Base "Truth" Dataset
truth_data = []
for _ in range(TOTAL_RECORDS):
    first_name = fake.first_name()
    last_name = fake.last_name()
    truth_data.append({
        "truth_id": str(uuid.uuid4()),
        "first_name": first_name,
        "last_name": last_name,
        "email": f"{first_name.lower()}.{last_name.lower()}@{fake.free_email_domain()}",
        "phone": fake.phone_number(),
        "address": fake.street_address(),
        "city": fake.city(),
        "state": fake.state_abbr(),
        "zip": fake.zipcode()
    })

truth_df = pd.DataFrame(truth_data)

# ---------------------------------------------------------
# 2. Generate Shopify Data (Sales - Highly Accurate, ~80% overlap)
# ---------------------------------------------------------
print("Generating Shopify data (Clean)...")
shopify_df = truth_df.sample(frac=0.80, random_state=1).copy()
shopify_df['shopify_id'] = [f"SHP_{random.randint(10000, 99999)}" for _ in range(len(shopify_df))]
# Shopify data remains mostly clean as it is transactional.
shopify_df = shopify_df[['shopify_id', 'first_name', 'last_name', 'email', 'phone', 'address', 'city', 'state', 'zip']]


# ---------------------------------------------------------
# 3. Generate Salesforce Data (CRM - Messy, ~60% overlap)
# ---------------------------------------------------------
print("Generating Salesforce data (Messy)...")
salesforce_df = truth_df.sample(frac=0.60, random_state=2).copy()
salesforce_df['sfdc_id'] = [f"003{fake.hexify(text='^^^^^^^^^^')}" for _ in range(len(salesforce_df))]

# INTENTIONAL MESSINESS:
def mess_up_sfdc(row):
    # 10% chance to drop the last name entirely
    if random.random() < 0.10:
        row['last_name'] = ""
    # 20% chance to introduce a typo in the email (e.g., .con instead of .com)
    if random.random() < 0.20:
        row['email'] = row['email'].replace(".com", ".con").replace(".net", ".nt")
    # 30% chance to strip formatting from phone numbers
    if random.random() < 0.30:
        row['phone'] = ''.join(filter(str.isdigit, str(row['phone'])))
    return row

salesforce_df = salesforce_df.apply(mess_up_sfdc, axis=1)
salesforce_df = salesforce_df[['sfdc_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state']]


# ---------------------------------------------------------
# 4. Generate Zendesk Data (Support - Alternative Formats, ~30% overlap)
# ---------------------------------------------------------
print("Generating Zendesk data (Alternative Formats)...")
zendesk_df = truth_df.sample(frac=0.30, random_state=3).copy()
zendesk_df['zd_user_id'] = [random.randint(1000000, 9999999) for _ in range(len(zendesk_df))]

# INTENTIONAL MESSINESS:
def mess_up_zendesk(row):
    # 40% chance to use a nickname (just the first 3 letters of the first name)
    if random.random() < 0.40 and len(row['first_name']) > 3:
        row['first_name'] = row['first_name'][:3]
    # 30% chance to use an alternative email address
    if random.random() < 0.30:
        row['email'] = f"contact_{row['first_name'].lower()}@{fake.free_email_domain()}"
    return row

zendesk_df = zendesk_df.apply(mess_up_zendesk, axis=1)
# Zendesk typically might not capture full addresses
zendesk_df = zendesk_df[['zd_user_id', 'first_name', 'last_name', 'email']]


# ---------------------------------------------------------
# 5. Export / API Push Boilerplate
# ---------------------------------------------------------
print("Saving simulated datasets to CSV...")
shopify_df.to_csv("shopify_customers_mock.csv", index=False)
salesforce_df.to_csv("salesforce_leads_mock.csv", index=False)
zendesk_df.to_csv("zendesk_users_mock.csv", index=False)

print("Data generation complete!")

config_path ='/Workspace/Users/kortum.facturas@gmail.com/Drafts/GAMBILL PROJECTS/3RD PROJECT/config.json'

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)

SHOPIFY_STORE = config.get("SHOPIFY_STORE")
SHOPIFY_ACCESS_TOKEN = config.get("X-Shopify-Access-Token")

SFDC_DOMAIN = config.get("SALESFORCE")
SFDC_DOMAIN_ACCESS_TOKEN = config.get("SALESFORCE_API_KEY")
SFDC_DOMAIN_ACCESS_TOKEN_SECRET = config.get("SALESFORCE_API_KEY_SECRET")

ZENDESK_SUBDOMAIN = config.get("ZENDESK")
ZENDESK_SUBDOMAIN_TOKEN = config.get("ZENDESK_API_KEY")


# """
# --- API PUSH BOILERPLATE (For mathew to implement) ---
# Once the data is generated, mathew can iterate through the dataframes 
# and push them to her developer accounts.

import requests
from requests.auth import HTTPBasicAuth

def push_to_shopify(df):
    url = f"https://{SHOPIFY_STORE}.myshopify.com/admin/api/2023-10/customers.json"
    headers = {"X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN, "Content-Type": "application/json"}
    
    for index, row in df.iterrows():
        payload = {
            "customer": {
                "first_name": row['first_name'],
                "last_name": row['last_name'],
                "email": row['email'],
                "phone": row['phone']
            }
        }
        response = requests.post(url, json=payload, headers=headers)
        print(f"Shopify Status: {response.status_code}")

def push_to_salesforce(df):
    # Requires an OAuth Access Token generated prior to this step
    url = f"https://{SFDC_DOMAIN}.lightning.force.com/services/data/v60.0/sobjects/Lead/"
    headers = {"Authorization": f"Bearer {SFDC_DOMAIN_ACCESS_TOKEN}", "Content-Type": "application/json"}
    
    for index, row in df.iterrows():
        payload = {
            "FirstName": row['first_name'],
            "LastName": row['last_name'] if row['last_name'] else "Unknown", # SFDC Lead requires LastName
            "Email": row['email'],
            "Phone": row['phone'],
            "City": row['city'],
            "State": row['state'],
            "Company": "B2C Customer" # SFDC Lead requires a Company field
        }
        response = requests.post(url, json=payload, headers=headers)
        print(f"Salesforce Status: {response.status_code}")

def push_to_zendesk(df):
    # Zendesk's create_or_update endpoint is perfect for bulk mock inserts
    url = f"https://{ZENDESK_SUBDOMAIN}.zendesk.com/api/v2/users/create_or_update.json"
    
    # Zendesk uses Basic Auth: email address appended with '/token' as the username, and the API token as the password.
    auth = HTTPBasicAuth("kortum.facturas@gmail.com/token", ZENDESK_SUBDOMAIN_TOKEN)
    headers = {"Content-Type": "application/json"}
    
    for index, row in df.iterrows():
        # Zendesk expects a single "name" field rather than separate first/last names
        full_name = f"{row['first_name']} {row['last_name']}".strip()
        
        payload = {
            "user": {
                "name": full_name if full_name else "Unknown User",
                "email": row['email']
            }
        }
        response = requests.post(url, json=payload, auth=auth, headers=headers)
        print(f"Zendesk Status: {response.status_code}")

# Example Execution (Pushing 100 records to test the API rate limits before larger batches)
push_to_shopify(shopify_df.head(10)) 
push_to_salesforce(salesforce_df.head(10))
push_to_zendesk(zendesk_df.head(10))
# """

In [0]:
# Debug: Check what credentials are loaded
print("SHOPIFY_STORE:", SHOPIFY_STORE)
print("SHOPIFY_ACCESS_TOKEN:", SHOPIFY_ACCESS_TOKEN[:10] + "..." if SHOPIFY_ACCESS_TOKEN else "None")
print()
print("SFDC_DOMAIN:", SFDC_DOMAIN)
print("SFDC_DOMAIN_ACCESS_TOKEN:", SFDC_DOMAIN_ACCESS_TOKEN[:10] + "..." if SFDC_DOMAIN_ACCESS_TOKEN else "None")
print()
print("ZENDESK_SUBDOMAIN:", ZENDESK_SUBDOMAIN)
print("ZENDESK_SUBDOMAIN_TOKEN:", ZENDESK_SUBDOMAIN_TOKEN[:10] + "..." if ZENDESK_SUBDOMAIN_TOKEN else "None")